# Phase 10 — Clean-v3 baseline comparison

Reconstructed notebook for `phase_10_clean_v3_comparison`. Recovers the paired per-case comparison across the four KUB baselines.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
paired = pd.read_csv(REPO / "reports/tables/phase10_clean_v3_paired_val_case_metrics.csv")
print("paired cases:", len(paired))
models = ["unet", "attention_unet", "unetpp", "deeplabv3plus"]
summary = pd.DataFrame([{
    "model": m,
    "n": int(paired[f"{m}_dice"].notna().sum()),
    "dice_mean": round(float(paired[f"{m}_dice"].mean()), 6),
    "dice_median": round(float(paired[f"{m}_dice"].median()), 6),
    "iou_mean": round(float(paired[f"{m}_iou"].mean()), 6),
} for m in models])
summary

paired cases: 280


,model,n,dice_mean,dice_median,iou_mean
0,unet,280,0.740613,0.823491,0.625382
1,attention_unet,280,0.752494,0.822223,0.634922
2,unetpp,280,0.764160,0.836520,0.652545
3,deeplabv3plus,280,0.750909,0.838469,0.646046


In [4]:
# Paired differences against the strongest baseline (DeepLabV3+).
import numpy as np
base = "deeplabv3plus"
rows = []
for m in models:
    if m == base:
        continue
    d = (paired[f"{m}_dice"] - paired[f"{base}_dice"]).dropna()
    rows.append({"model": m, "n_paired": len(d),
                 "mean_delta": round(float(d.mean()), 6),
                 "median_delta": round(float(d.median()), 6),
                 "n_better": int((d > 0).sum()), "n_worse": int((d < 0).sum())})
pd.DataFrame(rows)

,model,n_paired,mean_delta,median_delta,n_better,n_worse
0,unet,280,-0.010296,-0.011898,97,180
1,attention_unet,280,0.001585,-0.011491,114,164
2,unetpp,280,0.013251,-0.005592,120,156


In [5]:
# Per-case Dice distribution.
fig_, ax = plt.subplots(figsize=(6.2, 2.8))
data = [paired[f"{m}_dice"].dropna().values for m in models]
ax.violinplot(data, showmedians=True, widths=0.8)
ax.set_xticks(range(1, 5)); ax.set_xticklabels(models, rotation=15, fontsize=7)
ax.set_ylabel("Dice per case"); ax.set_title("Clean-v3 KUB baselines — per-case Dice")
plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_36992\962914730.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# Failure concentration: how many cases does each model lose entirely?
rows = []
for m in models:
    d = paired[f"{m}_dice"]
    rows.append({"model": m,
                 "cases_dice_below_0.5": int((d < 0.5).sum()),
                 "cases_dice_zero": int((d == 0).sum()),
                 "cases_above_0.9": int((d > 0.9).sum())})
pd.DataFrame(rows)

,model,cases_dice_below_0.5,cases_dice_zero,cases_above_0.9
0,unet,29,0,28
1,attention_unet,24,0,27
2,unetpp,23,0,45
3,deeplabv3plus,34,0,55


## Provenance

- `reports/tables/phase10_clean_v3_paired_val_case_metrics.csv`
- `reports/tables/phase10_final_all_models_epoch_results_long.csv`

In [7]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
